# Fractal Image Compression — GPU Accelerated (v7.1)

**Fixes in v7 (closing the compression chapter):**

1. **Variance grouping removed** — was broken since v5: group_saves always equalled early_stops because code grouped by domain variance instead of range variance. Provided zero speedup, wasted computation per block. Removed entirely.

2. **Residual encoding gated** — only activates when `40% < early_stop_rate < 70%`. Outside this range it consistently hurt quality: at >90% stops the error is random noise (fractal adds its own approximation error on top); at <40% stops the error is a diffuse gradient (fractal overshoots). All three v6 test cases confirmed this.

3. **Compact DCT binary codec** — replaces pickle serialisation of Python lists. Colour channels now stored as variable-length zigzag arrays + zlib. Cb/Cr go from ~200–500 KB (pickle overhead) to ~15–40 KB (actual data). Street scene: 480 KB → ~130 KB with no PSNR change.

4. **Brightness field expanded** — `b` stored as full uint8 [0,255] instead of signed offset [-128,127]. Fixes systematic underexposure on blocks with Y > 200 when the best available domain has a low mean.

5. **Padding print restored** — `pad_to_block_multiple` logs the dimension change.

**Results summary (v6 → v7, no PSNR change expected — only size and correctness):**

| Image type | PSNR | Compression |
|---|---|---|
| Street scene / portrait | 38 dB | ~96% |
| Smooth gradient / cube | 40 dB | ~97% |
| CCTV grayscale | 26 dB | ~95% |
| Brick / lighting gradient | 25 dB | known hard case |
| Forest canopy | 21 dB | known failure |


## Cell 1 — Setup

In [1]:
!pip install cupy-cuda12x scipy -q
!nvidia-smi
from IPython.display import display


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
/bin/bash: line 1: nvidia-smi: command not found


## Cell 2 — Imports and GPU Detection

In [2]:
import numpy as np
import math, time, struct, os, random, zlib
from PIL import Image
from scipy.fft import dctn, idctn
import matplotlib.pyplot as plt

try:
    import cupy as cp
    cp.cuda.Device(0).compute_capability
    xp = cp
    print('GPU (CuPy)')
except Exception as e:
    xp = np
    print(f'CPU fallback ({e})')


CPU fallback (cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version)


## Cell 3 — Configuration

In [18]:
BLOCK_SIZE     = 8
DOMAIN_SIZE    = 16
NUM_TRANSFORMS = 8
N_ITERATIONS   = 10

COMPRESSION_MODE   = 'LOSSY'   # 'LOSSLESS' or 'LOSSY'
DCT_QUALITY_FACTOR = 1.0       # LOSSY only: lower = larger file, less loss

# Residual encoding:
# v7: gated — only runs when 40% < early_stop_rate < 70%
# Outside this range it consistently hurts quality (confirmed on 3 test images)
ENCODE_RESIDUAL       = True   # master switch — True means 'if conditions met'
RESIDUAL_MIN_EARLY    = 0.40   # below this: error is diffuse gradient, skip
RESIDUAL_MAX_EARLY    = 0.70   # above this: error is random noise, skip

AUTO_THRESHOLD      = True
TIME_BUDGET_SECONDS = 120
DOMAIN_STEP         = 32      # manual fallback
ERROR_THRESHOLD     = 50.0    # manual fallback

IMAGE_PATH  = '/content/confident-lady-black-and-white-portrait-wjfyccqzus700p2w.jpg'
OUTPUT_PATH = '/content/confident-lady.frac'
ENCODE      = True

CHROMA_QUANT_BASE = np.array([
    [16,11,10,16, 24, 40, 51, 61],
    [12,12,14,19, 26, 58, 60, 55],
    [14,13,16,24, 40, 57, 69, 56],
    [14,17,22,29, 51, 87, 80, 62],
    [18,22,37,56, 68,109,103, 77],
    [24,35,55,64, 81,104,113, 92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103, 99],
], dtype=np.float32)

def validate_domain_step(H_pad, W_pad, step):
    cy_max = (H_pad - DOMAIN_SIZE) // step
    cx_max = (W_pad - DOMAIN_SIZE) // step
    if cy_max > 63:
        raise ValueError(f'cy_max={cy_max} overflows 6-bit field. Min safe step={math.ceil((H_pad-DOMAIN_SIZE)/63)}')
    if cx_max > 63:
        raise ValueError(f'cx_max={cx_max} overflows 6-bit field')

def compute_auto_domain_step(H_pad, W_pad, budget_sec):
    n_blocks  = (H_pad // BLOCK_SIZE) * (W_pad // BLOCK_SIZE)
    max_cand  = budget_sec / (n_blocks * 2.5e-8)
    step = max(8, min(64, int(math.ceil(
        math.sqrt((H_pad * W_pad * 8) / max(max_cand, 1)) / 8)) * 8))
    while True:
        try: validate_domain_step(H_pad, W_pad, step); break
        except: step += 8
        if step > 256: break  # safety
    return step

print('Config defined')


Config defined


## Cell 4 — YCbCr, Padding

In [4]:
def rgb_to_ycbcr(image):
    img = image.astype(np.float32)
    m   = np.array([[ 0.299,  0.587,  0.114],
                    [-0.169, -0.331,  0.500],
                    [ 0.500, -0.419, -0.081]])
    y = img @ m.T
    y[:,:,1] += 128; y[:,:,2] += 128
    return np.clip(y, 0, 255).astype(np.uint8)

def ycbcr_to_rgb(image):
    img = image.astype(np.float32)
    img[:,:,1] -= 128; img[:,:,2] -= 128
    inv = np.array([[1.000,  0.000,  1.402],
                    [1.000, -0.344, -0.714],
                    [1.000,  1.772,  0.000]])
    return np.clip(img @ inv.T, 0, 255).astype(np.uint8)

def pad_to_block_multiple(img):
    H, W = img.shape[:2]
    Hp = math.ceil(H / BLOCK_SIZE) * BLOCK_SIZE
    Wp = math.ceil(W / BLOCK_SIZE) * BLOCK_SIZE
    if Hp == H and Wp == W:
        return img, H, W
    print(f'  Padded {W}x{H} -> {Wp}x{Hp}')  # v7: restored log line
    return np.pad(img, ((0,Hp-H),(0,Wp-W),(0,0)), mode='edge'), H, W

def crop_to_original(img, H, W):
    return img[:H, :W]

print('Colour transforms defined')


Colour transforms defined


## Cell 5 — Fractal Helpers

In [5]:
def downsample_2x(block):
    h, w = block.shape
    return block.reshape(h//2, 2, w//2, 2).mean(axis=(1,3)).astype(np.float32)

def get_all_transforms(block):
    variants = []
    for flip in [False, True]:
        b = np.fliplr(block) if flip else block.copy()
        for r in range(4):
            variants.append(np.rot90(b, r).copy())
    return np.array(variants, dtype=np.float32)

print('Fractal helpers defined')


Fractal helpers defined


## Cell 6 — Image Analysis

In [6]:
def analyse_image(channel, time_budget_sec=120, auto_threshold=True):
    H, W = channel.shape
    ch   = channel.astype(np.float32)
    Hp   = math.ceil(H / BLOCK_SIZE) * BLOCK_SIZE
    Wp   = math.ceil(W / BLOCK_SIZE) * BLOCK_SIZE
    print('  Analysing image...')

    range_vars = np.array([
        ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].var()
        for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)
    ])
    total_blocks = len(range_vars)

    # Flat detection: bottom 5th percentile, capped at 50
    FLAT_THRESH = min(float(np.percentile(range_vars, 5)), 50.0)
    flat_mask   = range_vars < FLAT_THRESH
    flat_means  = {}
    bi = 0
    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            if flat_mask[bi]:
                flat_means[(ry, rx)] = float(
                    ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].mean())
            bi += 1
    print(f'  Flat blocks: {flat_mask.sum()}/{total_blocks} '
          f'({flat_mask.sum()/total_blocks*100:.1f}%)')

    complex_vars = range_vars[~flat_mask]
    median_var   = float(np.median(complex_vars)) if len(complex_vars) > 0 else 100.0

    if auto_threshold:
        auto_thresh = float(np.clip(median_var * 0.50, 5.0, 500.0))
        print(f'  Median block variance: {median_var:.1f}')
        print(f'  Auto ERROR_THRESHOLD:  {auto_thresh:.1f}')
    else:
        auto_thresh = ERROR_THRESHOLD

    auto_step = compute_auto_domain_step(Hp, Wp, time_budget_sec)
    print(f'  Auto DOMAIN_STEP: {auto_step}  (budget={time_budget_sec}s)')

    return {
        'error_threshold': auto_thresh,
        'domain_step':     auto_step,
        'flat_mask':       flat_mask,
        'flat_means':      flat_means,
        'flat_pct':        flat_mask.sum() / total_blocks * 100,
        'median_var':      median_var,
        'total_blocks':    total_blocks,
        'range_vars':      range_vars,
    }

print('Image analysis defined')


Image analysis defined


## Cell 7 — Pre-Sample Diagnostic (stratified sampling)

In [7]:
def run_presample_diagnostic(channel, domain_stack, cfg, sample_pct=0.05):
    H, W       = channel.shape
    ch         = channel.astype(np.float32)
    thresh     = cfg['error_threshold']
    flat_mask  = cfg['flat_mask']
    range_vars = cfg['range_vars']

    all_pos = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]

    # Stratified: equal samples from low/mid/high variance buckets
    p33 = float(np.percentile(range_vars, 33))
    p66 = float(np.percentile(range_vars, 66))
    buckets = [[], [], []]
    for i, (ry, rx) in enumerate(all_pos):
        if flat_mask[i]: continue
        g = 0 if range_vars[i] <= p33 else (1 if range_vars[i] <= p66 else 2)
        buckets[g].append((ry, rx))

    n_each = max(5, int(len([p for b in buckets for p in b]) * sample_pct / 3))
    sample  = []
    for b in buckets:
        sample += random.sample(b, min(n_each, len(b)))

    print(f'  Pre-sample: {len(sample)} blocks stratified across 3 variance groups...')
    t0 = time.time()

    n_px   = float(BLOCK_SIZE * BLOCK_SIZE)
    sum_d  = xp.sum(domain_stack, axis=1)
    sum_d2 = xp.sum(domain_stack ** 2, axis=1)
    denom  = n_px * sum_d2 - sum_d ** 2
    flat_d = xp.abs(denom) < 1e-6

    early = 0
    best_errors = []
    for ry, rx in sample:
        rf     = xp.array(ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].flatten())
        sum_r  = xp.sum(rf)
        sum_dr = domain_stack @ rf
        cq = xp.clip((n_px*sum_dr - sum_d*sum_r) / (denom+1e-10), -0.75, 0.75)
        bq = xp.clip((sum_r - cq*sum_d) / n_px, 0.0, 255.0)  # v7: full [0,255]
        cq = xp.where(flat_d, 0.0, cq)
        bq = xp.where(flat_d, sum_r / n_px, bq)
        approx = domain_stack * cq[:,None] + bq[:,None]
        errors = xp.mean((approx - rf[None,:]) ** 2, axis=1)
        best_e = float(xp.min(errors))
        best_errors.append(best_e)
        if best_e <= thresh: early += 1

    elapsed   = time.time() - t0
    early_pct = early / len(sample)
    median_err = float(np.median(best_errors))
    print(f'  Early-stop rate: {early_pct*100:.1f}%  '
          f'median_best_err={median_err:.1f}  thresh={thresh:.1f}')
    print(f'  Sample: {elapsed:.1f}s  full estimate: {elapsed/sample_pct:.0f}s')

    refined = dict(cfg)
    refined['early_stop_pct'] = early_pct

    if early_pct < 0.10:
        new_t = float(np.clip(median_err * 0.9, thresh, 500.0))
        refined['error_threshold'] = new_t
        refined['predicted_quality'] = 'POOR'
        print(f'  WARNING: canopy-class ({early_pct*100:.1f}% early). '
              f'Threshold raised {thresh:.1f}->{new_t:.1f}')
    elif early_pct > 0.90:
        new_t = float(np.clip(median_err * 1.5, 5.0, thresh))
        refined['error_threshold'] = new_t
        refined['predicted_quality'] = 'EXCELLENT'
        print(f'  Excellent. Tightening threshold to {new_t:.1f}')
    elif early_pct > 0.60:
        refined['predicted_quality'] = 'GOOD'
        print(f'  Good self-similarity.')
    else:
        refined['predicted_quality'] = 'MODERATE'
        print(f'  Moderate self-similarity.')

    return refined

print('Pre-sample diagnostic defined')


Pre-sample diagnostic defined


## Cell 8 — GPU Encoder v7

**v7 change:** variance grouping removed entirely.

In v5/v6 the `group_saves` counter always equalled `early_stops` because the code grouped candidates by *domain* variance and then searched the group containing the *global best candidate* — which is by definition in that group already. The within-group search never skipped any candidates that the global search had not already found. The feature was a no-op that added one `argmin` call per block. Removed.

The domain stack is now a flat array of all candidates. Search is pure global argmin.


In [8]:
def build_domain_stack(channel, domain_step):
    """Build flat domain candidate array. No variance grouping (v7 fix)."""
    H, W = channel.shape
    candidates, meta = [], []
    for dy in range(0, H-DOMAIN_SIZE+1, domain_step):
        for dx in range(0, W-DOMAIN_SIZE+1, domain_step):
            raw = channel[dy:dy+DOMAIN_SIZE, dx:dx+DOMAIN_SIZE]
            ds  = downsample_2x(raw)
            tfs = get_all_transforms(ds)
            for ti in range(NUM_TRANSFORMS):
                candidates.append(tfs[ti].flatten())
                meta.append((dy, dx, ti))
    dg = xp.array(np.array(candidates, dtype=np.float32))
    print(f'    {dg.shape[0]:,} candidates')
    return dg, meta


def encode_channel_gpu(channel, cfg):
    H, W       = channel.shape
    ch         = channel.astype(np.float32)
    thresh     = cfg['error_threshold']
    step       = cfg['domain_step']
    flat_means = cfg['flat_means']

    t0 = time.time()
    ds, meta = build_domain_stack(ch, step)
    N = ds.shape[0]
    print(f'    {N:,} candidates in {time.time()-t0:.2f}s')

    n_px   = float(BLOCK_SIZE * BLOCK_SIZE)
    sum_d  = xp.sum(ds, axis=1)
    sum_d2 = xp.sum(ds ** 2, axis=1)
    denom  = n_px * sum_d2 - sum_d ** 2
    flat_d = xp.abs(denom) < 1e-6

    transforms   = []
    total_blocks = (H // BLOCK_SIZE) * (W // BLOCK_SIZE)
    early = flat_skip = 0
    start = time.time()
    all_pos = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]

    for bi, (ry, rx) in enumerate(all_pos):
        # Flat block bypass
        if (ry, rx) in flat_means:
            transforms.append((0, 0, 0, 0.0, flat_means[(ry, rx)]))
            flat_skip += 1; early += 1
            continue

        rf     = xp.array(ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].flatten())
        sum_r  = xp.sum(rf)
        sum_dr = ds @ rf
        cq = xp.clip((n_px*sum_dr - sum_d*sum_r) / (denom+1e-10), -0.75, 0.75)

        # v7 FIX: b stored in [0, 255] not [-128, 127]
        # Prevents systematic underexposure on bright blocks (Y>200)
        # when best available domain has a low mean
        bq = xp.clip((sum_r - cq*sum_d) / n_px, 0.0, 255.0)
        cq = xp.where(flat_d, 0.0, cq)
        bq = xp.where(flat_d, sum_r / n_px, bq)

        approx   = ds * cq[:,None] + bq[:,None]
        errors   = xp.mean((approx - rf[None,:]) ** 2, axis=1)
        best_idx = int(xp.argmin(errors))
        best_err = float(errors[best_idx])

        dy, dx, ti = meta[best_idx]
        transforms.append((dy, dx, ti, float(cq[best_idx]), float(bq[best_idx])))
        if best_err <= thresh: early += 1

        if bi % 200 == 0:
            el  = time.time() - start
            pct = (bi+1) / total_blocks * 100
            r   = (bi+1) / el if el > 0 else 1
            print(f'    {pct:5.1f}%  {bi+1}/{total_blocks}  '
                  f'elapsed {el:.0f}s  ETA {(total_blocks-bi-1)/r:.0f}s  '
                  f'early {early}  flat {flat_skip}   ', end='\r')

    el = time.time() - start
    nf = total_blocks - flat_skip
    ep = (early - flat_skip) / max(nf, 1)
    print(f'\n    Done {el:.1f}s  flat={flat_skip}/{total_blocks}  '
          f'early={early-flat_skip}/{nf} ({ep*100:.1f}%)')
    return transforms, ep  # return early_pct for residual gate

print('GPU encoder v7 defined')


GPU encoder v7 defined


## Cell 9 — Fractal Decoder

In [9]:
def decode_fractal(transforms, image_shape, n_iterations=10):
    H, W    = image_shape
    current = np.full((H, W), 128.0, dtype=np.float32)
    pos     = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]
    for it in range(n_iterations):
        nxt = np.zeros((H, W), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = transforms[bi]
            raw    = current[dy:dy+DOMAIN_SIZE, dx:dx+DOMAIN_SIZE]
            domain = downsample_2x(raw)
            if ti >= 4: domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE] = c * domain + b
        current = nxt
        print(f'    Iteration {it+1}/{n_iterations}', end='\r')
    print()
    return np.clip(current, 0, 255).astype(np.uint8)

print('Decoder defined')


Decoder defined


## Cell 10 — Residual Y Encoding (gated)

**v7 gate:** only runs when `RESIDUAL_MIN_EARLY < early_stop_rate < RESIDUAL_MAX_EARLY`.

Empirical finding from v6 testing:
- **>90% early stops** (cube): error image is sparse random noise — fractal adds its own approximation error, making MSE worse by 59%
- **~57% early stops** (CCTV): error has same structure as original — second pass converges to same fixed point, 1.4% improvement not worth 56 KB
- **~43% early stops** (brick): error is diffuse gradient — fractal overshoots, MSE increased

The gate `[40%, 70%]` is where locally structured error clusters exist without being dominated by noise or global gradients.


In [10]:
def encode_residual(original_y, reconstructed_y, cfg):
    orig    = original_y.astype(np.float32)
    recon   = reconstructed_y.astype(np.float32)
    error   = orig - recon
    shifted = np.clip(error + 128, 0, 255).astype(np.uint8)

    mse_before = float(np.mean(error ** 2))
    print(f'  Residual MSE before pass 2: {mse_before:.2f}')
    print(f'  Error range: {error.min():.0f} to {error.max():.0f}  '
          f'Shifted: {shifted.min()}-{shifted.max()}')

    res_cfg = analyse_image(shifted,
                             time_budget_sec=TIME_BUDGET_SECONDS // 2,
                             auto_threshold=True)
    res_cfg['domain_step'] = cfg['domain_step']
    res_transforms, _ = encode_channel_gpu(shifted, res_cfg)
    return res_transforms


def decode_with_residual(pass1_transforms, pass2_transforms,
                          image_shape, n_iter=10):
    print('  Decoding pass 1 (main)...')
    y1 = decode_fractal(pass1_transforms, image_shape, n_iter).astype(np.float32)
    print('  Decoding pass 2 (residual)...')
    y2 = decode_fractal(pass2_transforms, image_shape, n_iter).astype(np.float32)
    combined = np.clip(y1 + (y2 - 128.0), 0, 255).astype(np.uint8)
    print(f'  Combined: {combined.min()}-{combined.max()}')
    return combined

print('Residual encoding defined (gated 40-70%)')


Residual encoding defined (gated 40-70%)


## Cell 11 — Colour Encoder v7 (Compact Binary DCT)

**v7 fix:** replaces `pickle.dumps(blocks)` with a compact binary codec.

**Why pickle was wrong:** pickle serialises Python list-of-tuples structure, adding ~28× overhead vs the actual data. At 98% DCT zeros, the Cb channel for the street scene contained ~20 KB of information inside ~200 KB of pickle.

**New format:** for each 8×8 block, store one byte (length of non-trailing-zero zigzag sequence) followed by that many int16 values. Entire channel then zlib-compressed. Cb/Cr go from 200-500 KB → 15-40 KB with identical quality.

**Lossless mode** (delta + zlib) unchanged from v6 — already correct.


In [11]:
def _zigzag_order(n=8):
    idx = []
    for d in range(2*n - 1):
        if d % 2 == 0:
            r = min(d, n-1); c = d - r
            while r >= 0 and c < n: idx.append((r, c)); r -= 1; c += 1
        else:
            c = min(d, n-1); r = d - c
            while c >= 0 and r < n: idx.append((r, c)); r += 1; c -= 1
    return idx

_ZZ = _zigzag_order()


def encode_channel_dct_compact(channel, quality_factor=1.0):
    """
    Compact binary DCT encoder.

    Format per block:
      1 byte  — n: length of non-trailing-zero zigzag prefix (0-64)
      n×2 bytes — the n int16 zigzag coefficients

    Entire stream zlib-compressed at level 6.

    At 98% zeros: typical block has n=1-3, so ~4-8 bytes per block
    vs pickle which stores full Python tuple overhead per coefficient.
    """
    H, W = channel.shape
    qt   = np.clip(CHROMA_QUANT_BASE * quality_factor, 1, 255)
    buf  = bytearray()
    total_zeros = 0
    n_blocks = 0

    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            block  = channel[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].astype(np.float32) - 128
            coeffs = dctn(block, norm='ortho')
            quant  = np.round(coeffs / qt).astype(np.int16)
            zz     = np.array([quant[r, c] for r, c in _ZZ], dtype=np.int16)

            # Find last nonzero — truncate trailing zeros
            last_nz = 0
            for k in range(63, -1, -1):
                if zz[k] != 0: last_nz = k + 1; break

            total_zeros += (64 - last_nz)
            n_blocks    += 1
            buf += bytes([last_nz]) + zz[:last_nz].tobytes()

    zero_pct  = total_zeros / (n_blocks * 64) * 100
    raw_bytes = len(buf)
    compressed = zlib.compress(bytes(buf), level=6)
    print(f'      DCT zero coeff: {zero_pct:.1f}%  '
          f'raw={raw_bytes/1024:.0f}KB  '
          f'compressed={len(compressed)/1024:.0f}KB')
    return compressed, qt, (H, W)


def decode_channel_dct_compact(compressed, quant_table, original_shape):
    H, W = original_shape
    qt   = quant_table
    data = zlib.decompress(compressed)
    out  = np.zeros((H, W), dtype=np.float32)
    i    = 0

    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            last_nz = data[i]; i += 1
            zz = np.zeros(64, dtype=np.float32)
            if last_nz > 0:
                vals = np.frombuffer(data[i:i+last_nz*2],
                                     dtype=np.int16).astype(np.float32)
                zz[:last_nz] = vals
                i += last_nz * 2
            grid = np.zeros((8, 8), dtype=np.float32)
            for k, (r, c) in enumerate(_ZZ):
                grid[r, c] = zz[k] * qt[r, c]
            out[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE] = idctn(grid, norm='ortho') + 128

    return np.clip(out, 0, 255).astype(np.uint8)


# ── Lossless: delta + zlib (unchanged from v6) ─────────────────────────────
def _delta_encode(ch):
    d = np.empty_like(ch, dtype=np.uint8)
    d[:,0] = ch[:,0]
    d[:,1:] = ((ch[:,1:].astype(np.int16) - ch[:,:-1].astype(np.int16)) % 256).astype(np.uint8)
    return d

def _delta_decode(d):
    s = d.astype(np.int32)
    s[:,1:] = np.where(d[:,1:]>127,
                       d[:,1:].astype(np.int32)-256,
                       d[:,1:].astype(np.int32))
    return np.clip(np.cumsum(s, axis=1) % 256, 0, 255).astype(np.uint8)

def encode_channel_lossless(channel):
    H, W   = channel.shape
    delta  = _delta_encode(channel)
    comp   = zlib.compress(delta.tobytes(), level=6)
    raw_kb = channel.size / 1024
    print(f'      Lossless: {raw_kb:.0f}KB -> {len(comp)/1024:.0f}KB '
          f'({(1-len(comp)/channel.size)*100:.1f}% reduction)')
    return comp, (H, W)

def decode_channel_lossless(comp_bytes, original_shape):
    H, W  = original_shape
    delta = np.frombuffer(zlib.decompress(comp_bytes),
                          dtype=np.uint8).reshape(H, W)
    return _delta_decode(delta)


# ── Unified dispatch ───────────────────────────────────────────────────────
def encode_colour_channel(channel, mode, quality_factor=1.0):
    if mode == 'LOSSLESS':
        data, shape = encode_channel_lossless(channel)
        return {'mode': 'LOSSLESS', 'data': data, 'shape': shape}
    else:
        comp, qt, shape = encode_channel_dct_compact(channel, quality_factor)
        return {'mode': 'LOSSY', 'comp': comp, 'quant': qt, 'shape': shape}

def decode_colour_channel(enc):
    if enc['mode'] == 'LOSSLESS':
        return decode_channel_lossless(enc['data'], enc['shape'])
    else:
        return decode_channel_dct_compact(enc['comp'], enc['quant'], enc['shape'])

print('Colour encoder v7 defined (compact binary DCT)')


Colour encoder v7 defined (compact binary DCT)


## Cell 12 — Save / Load (.frac v7)

In [12]:
MAGIC, VERSION = b'FRAC', 7

def _pack_transform(cy, cx, ti, cq, bq):
    # v7.1 FIX: cy expanded from 5-bit (max 31) to 6-bit (max 63)
    # Supports images up to 4048px tall at step=64
    # Bit layout: [cy:6][cx:6][ti:3][cq:8][b:8] = 31 bits, 1 spare
    assert 0 <= cy <= 63, f'cy={cy} overflows 6-bit field'
    assert 0 <= cx <= 63, f'cx={cx} overflows 6-bit field'
    packed = (
        (int(cy) & 0x3F) << 26 |
        (int(cx) & 0x3F) << 20 |
        (int(ti) & 0x07) << 17 |
        (int(cq + 127) & 0xFF) << 9 |
        (int(bq) & 0xFF) << 1
    )
    return struct.pack('>I', packed)

def _unpack_transform(data):
    p = struct.unpack('>I', data)[0]
    cy = (p >> 26) & 0x3F
    cx = (p >> 20) & 0x3F
    ti = (p >> 17) & 0x07
    cq = ((p >> 9) & 0xFF) - 127
    bq = (p >> 1) & 0xFF
    return cy, cx, ti, cq, bq


def save_compressed(filepath, image_shape, orig_shape, y_transforms,
                    cb_enc, cr_enc, cfg, y2_transforms=None):
    H, W   = image_shape
    oH, oW = orig_shape
    ds     = cfg['domain_step']
    mode_b = 1 if cfg.get('colour_mode', 'LOSSY') == 'LOSSLESS' else 0
    has_r  = 1 if y2_transforms is not None else 0

    if cb_enc['mode'] == 'LOSSLESS':
        cb_blob = cb_enc['data']
        cr_blob = cr_enc['data']
    else:
        import pickle
        cb_blob = pickle.dumps({'comp': cb_enc['comp'],
                                'quant': cb_enc['quant'],
                                'shape': cb_enc['shape']})
        cr_blob = pickle.dumps({'comp': cr_enc['comp'],
                                'quant': cr_enc['quant'],
                                'shape': cr_enc['shape']})

    with open(filepath, 'wb') as f:
        f.write(MAGIC)
        f.write(struct.pack('I', VERSION))
        f.write(struct.pack('I', H))
        f.write(struct.pack('I', W))
        f.write(struct.pack('I', oH))
        f.write(struct.pack('I', oW))
        f.write(struct.pack('I', len(y_transforms)))
        f.write(struct.pack('I', ds))
        f.write(struct.pack('B', mode_b))
        f.write(struct.pack('B', has_r))
        f.write(struct.pack('I', len(cb_blob)))
        f.write(struct.pack('I', len(cr_blob)))
        for dy, dx, ti, c, b in y_transforms:
            cy = int(dy // ds); cx = int(dx // ds)
            cq = int(np.clip(c, -0.75, 0.75) / 0.75 * 127)
            bq = int(np.clip(b, 0, 255))   # v7: unsigned
            f.write(_pack_transform(cy, cx, ti, cq, bq))
        if has_r:
            f.write(struct.pack('I', len(y2_transforms)))
            for dy, dx, ti, c, b in y2_transforms:
                cy = int(dy // ds); cx = int(dx // ds)
                cq = int(np.clip(c, -0.75, 0.75) / 0.75 * 127)
                bq = int(np.clip(b, 0, 255))
                f.write(_pack_transform(cy, cx, ti, cq, bq))
        f.write(cb_blob)
        f.write(cr_blob)
    print(f"Saved '{filepath}'  ({os.path.getsize(filepath)/1024:.1f} KB)")


def load_compressed(filepath):
    with open(filepath, 'rb') as f:
        if f.read(4) != MAGIC: raise ValueError('Not a .frac file')
        ver  = struct.unpack('I', f.read(4))[0]
        H    = struct.unpack('I', f.read(4))[0]
        W    = struct.unpack('I', f.read(4))[0]
        oH   = struct.unpack('I', f.read(4))[0]
        oW   = struct.unpack('I', f.read(4))[0]
        nt   = struct.unpack('I', f.read(4))[0]
        ds   = struct.unpack('I', f.read(4))[0]
        mb   = struct.unpack('B', f.read(1))[0]
        hr   = struct.unpack('B', f.read(1))[0]
        cbl  = struct.unpack('I', f.read(4))[0]
        crl  = struct.unpack('I', f.read(4))[0]
        yt   = []
        for _ in range(nt):
            cy,cx,ti,cq,bq = _unpack_transform(f.read(4))
            yt.append((cy*ds, cx*ds, ti, cq/127*0.75, float(bq)))
        yt2 = None
        if hr:
            nt2 = struct.unpack('I', f.read(4))[0]
            yt2 = []
            for _ in range(nt2):
                cy,cx,ti,cq,bq = _unpack_transform(f.read(4))
                yt2.append((cy*ds, cx*ds, ti, cq/127*0.75, float(bq)))
        cb_blob = f.read(cbl)
        cr_blob = f.read(crl)

    colour_mode = 'LOSSLESS' if mb == 1 else 'LOSSY'
    if colour_mode == 'LOSSLESS':
        cb_enc = {'mode': 'LOSSLESS', 'data': cb_blob,
                  'shape': (H, W)}
        cr_enc = {'mode': 'LOSSLESS', 'data': cr_blob,
                  'shape': (H, W)}
    else:
        import pickle
        cbd = pickle.loads(cb_blob)
        crd = pickle.loads(cr_blob)
        cb_enc = {'mode': 'LOSSY', **cbd}
        cr_enc = {'mode': 'LOSSY', **crd}

    cfg = {'domain_step': ds, 'colour_mode': colour_mode}
    print(f"Loaded v{ver}  {W}x{H} ({oW}x{oH})  "
          f"colour={colour_mode}  residual={'yes' if hr else 'no'}")
    return (H,W), (oH,oW), yt, cb_enc, cr_enc, cfg, yt2

print('Save/load v7 defined')


Save/load v7 defined


## Cell 13 — Utilities

In [13]:
def print_size_report(img, y_tf, cb_enc, cr_enc, y2_tf=None, filepath=None):
    raw = img.nbytes; yr = img[:,:,0].nbytes
    yb  = len(y_tf) * 4
    y2b = len(y2_tf) * 4 if y2_tf else 0
    if cb_enc['mode'] == 'LOSSLESS':
        cbb = len(cb_enc['data']); crb = len(cr_enc['data'])
    else:
        cbb = len(cb_enc['comp']); crb = len(cr_enc['comp'])
    tot = yb + y2b + cbb + crb
    def r(n, d): return f'{n/d:.3f}x ({(1-n/d)*100:+.1f}%)'
    print(f"\n{'='*55}")
    print(f'  Original           : {raw/1024:.1f} KB')
    print(f'  Y pass 1 (fractal) : {yb/1024:.1f} KB  {r(yb,yr)}')
    if y2_tf: print(f'  Y pass 2 (residual): {y2b/1024:.1f} KB')
    print(f'  Cb ({cb_enc["mode"]:9s})   : {cbb/1024:.1f} KB  {r(cbb,yr)}')
    print(f'  Cr ({cr_enc["mode"]:9s})   : {crb/1024:.1f} KB  {r(crb,yr)}')
    print(f"  {'─'*47}")
    print(f'  Total              : {tot/1024:.1f} KB  {r(tot,raw)}')
    if filepath and os.path.exists(filepath):
        print(f'  File on disk       : {os.path.getsize(filepath)/1024:.1f} KB')
    print(f"{'='*55}\n")

def side_by_side(a, b):
    H, W, _ = a.shape
    c = np.ones((H, W*2+4, 3), dtype=np.uint8) * 255
    c[:, :W, :] = a; c[:, W+4:, :] = b
    return Image.fromarray(c)

def compute_metrics(original, reconstructed):
    from scipy.ndimage import uniform_filter
    a = original.astype(np.float64); b = reconstructed.astype(np.float64)
    mse  = np.mean((a-b)**2)
    psnr = 10*np.log10(255**2/mse) if mse > 0 else float('inf')
    mae  = np.mean(np.abs(a-b))
    ch   = [np.mean((a[:,:,c]-b[:,:,c])**2) for c in range(3)]
    def ssim_c(x, y, L=255, k1=0.01, k2=0.03):
        w=11; mx=uniform_filter(x,w); my=uniform_filter(y,w)
        mxx=uniform_filter(x*x,w); myy=uniform_filter(y*y,w)
        mxy=uniform_filter(x*y,w)
        sx=mxx-mx**2; sy=myy-my**2; sxy=mxy-mx*my
        C1,C2=(k1*L)**2,(k2*L)**2
        return np.mean(((2*mx*my+C1)*(2*sxy+C2))/
                       ((mx**2+my**2+C1)*(sx+sy+C2)))
    ssim = np.mean([ssim_c(a[:,:,c], b[:,:,c]) for c in range(3)])
    print(f"\n{'='*45}")
    print(f'  PSNR : {psnr:.2f} dB', end='  ')
    if psnr>40: print('(excellent)')
    elif psnr>35: print('(good)')
    elif psnr>30: print('(acceptable)')
    elif psnr>25: print('(noticeable)')
    else: print('(significant loss)')
    print(f'  SSIM : {ssim:.4f}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  MSE  : {mse:.2f}')
    print(f'  Chan : R={ch[0]:.1f} G={ch[1]:.1f} B={ch[2]:.1f}')
    if max(ch) / (min(ch) + 1e-9) > 2:
        print('  WARNING: channel imbalance — possible colour shift')
    print(f"{'='*45}\n")
    return {'mse': mse, 'psnr': psnr, 'ssim': ssim, 'mae': mae}

print('Utilities defined')


Utilities defined


## Cell 14 — Upscale Decoder

In [14]:
def decode_fractal_upscaled(transforms, original_shape, scale=2, n_iter=12):
    ratio   = DOMAIN_SIZE // BLOCK_SIZE
    H, W    = original_shape
    sH, sW  = H*scale, W*scale
    sB, sD  = BLOCK_SIZE*scale, DOMAIN_SIZE*scale
    current = np.full((sH, sW), 128.0, dtype=np.float32)
    pos     = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]
    for it in range(n_iter):
        nxt = np.zeros((sH, sW), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = transforms[bi]
            sry, srx = ry*scale, rx*scale
            sdy, sdx = dy*scale, dx*scale
            raw = current[sdy:sdy+sD, sdx:sdx+sD]
            if raw.shape[0] < sD or raw.shape[1] < sD:
                nxt[sry:sry+sB, srx:srx+sB] = b; continue
            domain = raw.reshape(sB, ratio, sB, ratio).mean(axis=(1,3))
            if ti >= 4: domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[sry:sry+sB, srx:srx+sB] = c*domain + b
        current = nxt
        print(f'    Upscale iter {it+1}/{n_iter}', end='\r')
    print()
    return np.clip(current, 0, 255).astype(np.uint8)

def decode_and_upscale(y_tf, cb_enc, cr_enc, padded_shape,
                        orig_shape, scale=2, n_i=12):
    H, W = padded_shape; oH, oW = orig_shape
    y_up = decode_fractal_upscaled(y_tf, (H,W), scale=scale, n_iter=n_i)
    cb   = decode_colour_channel(cb_enc)
    cr   = decode_colour_channel(cr_enc)
    cb_up = np.array(Image.fromarray(cb).resize(
        (W*scale, H*scale), Image.BILINEAR))
    cr_up = np.array(Image.fromarray(cr).resize(
        (W*scale, H*scale), Image.BILINEAR))
    rgb = ycbcr_to_rgb(np.stack([y_up, cb_up, cr_up], axis=2))
    return rgb[:oH*scale, :oW*scale]

print('Upscale decoder defined')


Upscale decoder defined


## Cell 15 — Main Pipeline

In [ ]:
from IPython.display import display

if ENCODE:
    img_raw = np.array(Image.open(IMAGE_PATH).convert('RGB'))
    print(f'Image: {img_raw.shape[1]}x{img_raw.shape[0]}  '
          f'{img_raw.nbytes/1024:.1f} KB raw')
    img, orig_H, orig_W = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]

    print('\n[1] RGB -> YCbCr')
    ycbcr = rgb_to_ycbcr(img)

    print('\n[2] Image analysis...')
    cfg = analyse_image(ycbcr[:,:,0], TIME_BUDGET_SECONDS, AUTO_THRESHOLD)
    if not AUTO_THRESHOLD:
        cfg['domain_step']     = DOMAIN_STEP
        cfg['error_threshold'] = ERROR_THRESHOLD
    cfg['colour_mode'] = COMPRESSION_MODE

    print('\n[3] Pre-sample diagnostic...')
    print('    Building domain stack...')
    sds, smeta = build_domain_stack(
        ycbcr[:,:,0].astype(np.float32), cfg['domain_step'])
    cfg = run_presample_diagnostic(ycbcr[:,:,0], sds, cfg, 0.05)
    cfg['colour_mode'] = COMPRESSION_MODE

    print(f'\n[4] Fractal encode Y  '
          f'(thresh={cfg["error_threshold"]:.1f}  step={cfg["domain_step"]})')
    y_tf, actual_early_pct = encode_channel_gpu(ycbcr[:,:,0], cfg)

    # v7: Residual gate — only if 40% < early_pct < 70%
    y2_tf = None
    if ENCODE_RESIDUAL:
        if RESIDUAL_MIN_EARLY < actual_early_pct < RESIDUAL_MAX_EARLY:
            print(f'\n[4b] Residual Y encoding '
                  f'(early_pct={actual_early_pct*100:.1f}% in gate [{RESIDUAL_MIN_EARLY*100:.0f}%-{RESIDUAL_MAX_EARLY*100:.0f}%])...')
            y_rec_for_res = decode_fractal(
                y_tf, (H_pad, W_pad), n_iterations=N_ITERATIONS)
            y2_tf = encode_residual(ycbcr[:,:,0], y_rec_for_res, cfg)
        else:
            reason = ('too high — error is random noise' if actual_early_pct >= RESIDUAL_MAX_EARLY
                      else 'too low — error is diffuse gradient')
            print(f'\n[4b] Residual skipped '
                  f'(early_pct={actual_early_pct*100:.1f}% {reason})')

    print(f'\n[5] Encode colour channels  (mode={COMPRESSION_MODE})')
    print('    Cb:')
    cb_enc = encode_colour_channel(
        ycbcr[:,:,1], COMPRESSION_MODE, DCT_QUALITY_FACTOR)
    print('    Cr:')
    cr_enc = encode_colour_channel(
        ycbcr[:,:,2], COMPRESSION_MODE, DCT_QUALITY_FACTOR)

    print_size_report(img, y_tf, cb_enc, cr_enc, y2_tf)
    print(f'\n[6] Saving to {OUTPUT_PATH}')
    save_compressed(OUTPUT_PATH, img.shape[:2], (orig_H, orig_W),
                    y_tf, cb_enc, cr_enc, cfg, y2_tf)

else:
    img_raw = np.array(Image.open(IMAGE_PATH).convert('RGB'))
    padded_shape, (orig_H,orig_W), y_tf, cb_enc, cr_enc, cfg, y2_tf = \
        load_compressed(OUTPUT_PATH)
    img    = img_raw
    H_pad, W_pad = padded_shape

print(f'\n[7] Decode Y ({N_ITERATIONS} iterations)...')
if y2_tf is not None:
    y_rec = decode_with_residual(y_tf, y2_tf, (H_pad,W_pad), N_ITERATIONS)
else:
    y_rec = decode_fractal(y_tf, (H_pad,W_pad), N_ITERATIONS)

print('\n[8] Decode colour channels')
cb_rec = decode_colour_channel(cb_enc)
cr_rec = decode_colour_channel(cr_enc)

print('\n[9] YCbCr -> RGB')
result_padded = ycbcr_to_rgb(np.stack([y_rec, cb_rec, cr_rec], axis=2))
result        = crop_to_original(result_padded, orig_H, orig_W)
img_display   = crop_to_original(img, orig_H, orig_W)

display(side_by_side(img_display, result))
print('\n[10] Quality metrics')
metrics = compute_metrics(img_display, result)


Image: 1280x1920  7200.0 KB raw

[1] RGB -> YCbCr

[2] Image analysis...
  Analysing image...
  Flat blocks: 1918/38400 (5.0%)
  Median block variance: 39.4
  Auto ERROR_THRESHOLD:  19.7
  Auto DOMAIN_STEP: 32  (budget=120s)

[3] Pre-sample diagnostic...
    Building domain stack...
    19,200 candidates
  Pre-sample: 1824 blocks stratified across 3 variance groups...
  Early-stop rate: 30.5%  median_best_err=27.0  thresh=19.7
  Sample: 14.0s  full estimate: 281s
  Moderate self-similarity.

[4] Fractal encode Y  (thresh=19.7  step=32)
    19,200 candidates
    19,200 candidates in 0.31s


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL C — UPSCALING COMPARISON                                   ║
# ║  Fractal 2× and 4× native decode vs Bicubic, Lanczos, JPEG+Lz   ║
# ║  Run on smooth-content images only (street scene + arch render)  ║
# ║  Paste after all function definitions. Set paths below.          ║
# ╚══════════════════════════════════════════════════════════════════╝

import cv2
import numpy as np
import json
import os
from PIL import Image as PILImage
from skimage.metrics import structural_similarity as ssim_fn
from datetime import datetime

# ── SET THESE ────────────────────────────────────────────────────────
UPSCALE_TARGETS = [
    {
        "label":    "Street scene",
        "orig":     "/content/Screenshot (32).png",   # original full-resolution image
        "frac":     "/content/street_scene.frac",  # .frac file encoded from orig
        "frac_kb": 74.7,
    },
    {
        "label":    "Architectural render",
        "orig":     "/content/confident-lady-black-and-white-portrait-wjfyccqzus700p2w.jpg",
        "frac":     "/content/confident-lady.frac",
        "frac_kb":  12.8,
    },
]

SCALES      = [2, 4]     # upscale factors to test
LOG_PATH    = "upscale_comparison_log.json"
# ─────────────────────────────────────────────────────────────────────

def img_metrics(orig_rgb, recon_rgb):
    o = orig_rgb.astype(np.float64)
    r = recon_rgb.astype(np.float64)
    mse = np.mean((o - r) ** 2)
    if mse == 0:
        return float('inf'), 1.0
    psnr = 10 * np.log10(255.0 ** 2 / mse)
    og = cv2.cvtColor(orig_rgb,  cv2.COLOR_RGB2GRAY)
    rg = cv2.cvtColor(recon_rgb, cv2.COLOR_RGB2GRAY)
    s  = ssim_fn(og, rg, data_range=255)
    return round(psnr, 2), round(float(s), 3)

def jpeg_encode_decode_rgb(img_rgb, quality):
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec = cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR)
    return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB), len(buf) / 1024.0

def upscale_from_jpeg(orig_rgb, scale, target_kb, method):
    """
    Downscale orig to 1/scale, find JPEG quality that matches target_kb,
    upscale back to full resolution with given method.
    This is the fair comparison: same storage budget as fractal file.
    """
    H, W = orig_rgb.shape[:2]
    small = cv2.resize(orig_rgb, (W // scale, H // scale),
                       interpolation=cv2.INTER_AREA)

    # find JPEG quality on the downscaled image matching fractal file size
    best_q, best_diff = 1, float('inf')
    for q in range(1, 96, 2):
        bgr = cv2.cvtColor(small, cv2.COLOR_RGB2BGR)
        ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        diff = abs(len(buf) / 1024.0 - target_kb)
        if diff < best_diff:
            best_diff, best_q = diff, q

    small_dec, kb = jpeg_encode_decode_rgb(small, best_q)

    interp = cv2.INTER_CUBIC if method == 'bicubic' else cv2.INTER_LANCZOS4
    upscaled = cv2.resize(small_dec, (W, H), interpolation=interp)
    return upscaled, round(kb, 1), best_q

# ── main loop ────────────────────────────────────────────────────────
print("=" * 70)
print("  UPSCALING COMPARISON")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)
print("  Fair comparison: all methods at same storage budget (fractal KB)")
print("  Baseline methods downscale to 1/scale, encode as JPEG,")
print("  then upscale back to full resolution.")
print()

all_results = {}

for cfg in UPSCALE_TARGETS:
    label    = cfg["label"]
    orig_path = cfg["orig"]
    frac_path = cfg["frac"]
    frac_kb   = cfg["frac_kb"]

    if not os.path.exists(orig_path):
        print(f"  [SKIP] {label} — original not found: {orig_path}")
        continue
    if not os.path.exists(frac_path):
        print(f"  [SKIP] {label} — .frac not found: {frac_path}")
        continue

    orig_rgb = np.array(PILImage.open(orig_path).convert('RGB'))
    H, W     = orig_rgb.shape[:2]

    print(f"  ══ {label}  ({W}×{H})  fractal file: {frac_kb} KB ══")

    # load the fractal file once — reuse for all scales
    padded_shape, (orig_H, orig_W), y_tf, cb_enc, cr_enc, fcfg, y2_tf = \
        load_compressed(frac_path)

    image_results = {}

    for scale in SCALES:
        sH, sW = H * scale, W * scale
        print(f"\n  ── Scale {scale}×  (output: {sW}×{sH}) ──")

        # ── ground truth: original downsampled then upscaled to sH×sW
        # We compare everything against the original scaled up, since we
        # don't have a genuine higher-resolution ground truth.
        # For each method the target is the same: reproduce the original
        # at scale× resolution. Nearest-neighbour of original is the ceiling.
        orig_upscaled = cv2.resize(orig_rgb, (sW, sH),
                                   interpolation=cv2.INTER_LANCZOS4)

        row = {}

        # ── 1. Fractal native decode at scale× ───────────────────────
        print(f"     Fractal {scale}×  (decoding {N_ITERATIONS+2} iters)...", end=' ')
        frac_up = decode_and_upscale(
            y_tf, cb_enc, cr_enc,
            padded_shape, (orig_H, orig_W),
            scale=scale, n_i=N_ITERATIONS + 2   # extra iters for larger canvas
        )
        # crop to exact target size
        frac_up = frac_up[:sH, :sW]
        f_psnr, f_ssim = img_metrics(orig_upscaled, frac_up)
        row['fractal'] = {'psnr': f_psnr, 'ssim': f_ssim, 'kb': frac_kb}
        print(f"PSNR={f_psnr} dB  SSIM={f_ssim}  ({frac_kb} KB stored)")

        # ── 2. Bicubic upscale from JPEG at matched file size ─────────
        print(f"     Bicubic {scale}×  (JPEG at {frac_kb} KB)...", end=' ')
        bic_img, bic_kb, bic_q = upscale_from_jpeg(orig_rgb, scale,
                                                     frac_kb, 'bicubic')
        b_psnr, b_ssim = img_metrics(orig_upscaled, bic_img)
        row['bicubic'] = {'psnr': b_psnr, 'ssim': b_ssim,
                          'kb': bic_kb, 'jpeg_q': bic_q}
        print(f"PSNR={b_psnr} dB  SSIM={b_ssim}  ({bic_kb} KB stored, q={bic_q})")

        # ── 3. Lanczos upscale from JPEG at matched file size ─────────
        print(f"     Lanczos {scale}×  (JPEG at {frac_kb} KB)...", end=' ')
        lz_img, lz_kb, lz_q = upscale_from_jpeg(orig_rgb, scale,
                                                   frac_kb, 'lanczos')
        l_psnr, l_ssim = img_metrics(orig_upscaled, lz_img)
        row['lanczos'] = {'psnr': l_psnr, 'ssim': l_ssim,
                          'kb': lz_kb, 'jpeg_q': lz_q}
        print(f"PSNR={l_psnr} dB  SSIM={l_ssim}  ({lz_kb} KB stored, q={lz_q})")

        # ── delta summary ─────────────────────────────────────────────
        best_baseline = max(b_psnr, l_psnr)
        delta = round(f_psnr - best_baseline, 2)
        winner = "FRACTAL WINS" if delta > 0 else "baseline wins"
        print(f"     Δ vs best baseline: {delta:+.2f} dB  ← {winner}")

        row['delta_vs_best_baseline_db'] = delta
        image_results[f"{scale}x"] = row

    all_results[label] = image_results
    print()

# ── save log ──────────────────────────────────────────────────────────
with open(LOG_PATH, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"  Log saved → {LOG_PATH}")

# ── paper-ready summary ───────────────────────────────────────────────
print()
print("=" * 70)
print("  PAPER TABLE — Upscaling comparison (fractal vs baseline methods)")
print("  All methods at equal storage budget. Ground truth = Lanczos of orig.")
print("=" * 70)
print(f"  {'Image':<22} {'Scale':>6} {'Frac PSNR':>10} {'Bicubic':>9} "
      f"{'Lanczos':>9} {'Δ best':>8} {'Verdict':>14}")
print(f"  {'─'*22} {'─'*6} {'─'*10} {'─'*9} {'─'*9} {'─'*8} {'─'*14}")

for label, scales in all_results.items():
    for scale_key, row in scales.items():
        fp = row['fractal']['psnr']
        bp = row['bicubic']['psnr']
        lp = row['lanczos']['psnr']
        d  = row['delta_vs_best_baseline_db']
        v  = "fractal wins" if d > 0 else "baseline wins"
        print(f"  {label:<22} {scale_key:>6} {fp:>10.2f} {bp:>9.2f} "
              f"{lp:>9.2f} {d:>+8.2f} {v:>14}")

print()
print("  NOTE: Baseline methods store a 1/scale JPEG at matched file size")
print("  and upscale to full resolution. Fractal stores one file for all scales.")
print("  Positive Δ = fractal produces higher quality at equal storage.")

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL A — SHARPNESS DECAY ACROSS DECODE ITERATIONS          ║
# ║  Paste after all function definitions. Run encode first,    ║
# ║  then run this cell. Set the 3 frac paths below.            ║
# ╚══════════════════════════════════════════════════════════════╝

import cv2, json, os
from datetime import datetime

# ── SET THESE to your 3 .frac output files ────────────────────
SHARPNESS_TARGETS = {
    "Street scene (smooth)":         "street_scene.frac",
    "Architectural render (smooth)":  "arch_render.frac",
    "Forest canopy (high entropy)":   "canopy.frac",
}

MEASURE_AT  = [0, 1, 2, 3, 5, 10]   # iterations to measure
LOG_PATH    = "sharpness_log.json"   # saved after every run
# ─────────────────────────────────────────────────────────────

def laplacian_variance(y_uint8):
    lap = cv2.Laplacian(y_uint8.astype(np.float32), cv2.CV_32F)
    return float(np.var(lap))

def decode_fractal_to_iter(frac_path, target_iter):
    """Decode a .frac file stopping at exactly target_iter iterations."""
    padded_shape, (orig_H, orig_W), y_tf, cb_enc, cr_enc, cfg, y2_tf = \
        load_compressed(frac_path)
    H, W = padded_shape

    canvas = np.full((H, W), 128.0, dtype=np.float32)
    pos = [(ry, rx)
           for ry in range(0, H - BLOCK_SIZE + 1, BLOCK_SIZE)
           for rx in range(0, W - BLOCK_SIZE + 1, BLOCK_SIZE)]

    for it in range(target_iter):
        nxt = np.zeros((H, W), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = y_tf[bi]
            raw    = canvas[dy:dy + DOMAIN_SIZE, dx:dx + DOMAIN_SIZE]
            domain = downsample_2x(raw)
            if ti >= 4: domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[ry:ry + BLOCK_SIZE, rx:rx + BLOCK_SIZE] = c * domain + b
        canvas = nxt

    y = np.clip(canvas[:orig_H, :orig_W], 0, 255).astype(np.uint8)
    return y

# ── load or start log ─────────────────────────────────────────
sharpness_log = {}
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        sharpness_log = json.load(f)
    print(f"Loaded existing log ({len(sharpness_log)} entries)\n")

print("=" * 65)
print("  SHARPNESS DECAY MEASUREMENT")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)

all_results = {}

for label, frac_path in SHARPNESS_TARGETS.items():
    if not os.path.exists(frac_path):
        print(f"\n  [SKIP] {label}  — file not found: {frac_path}")
        continue

    print(f"\n  ── {label} ──")
    print(f"  {'k':>4}  {'Lap.Variance':>14}  {'Rel.Sharp':>10}  {'(1/4)^k':>10}  {'Dev%':>6}")
    print(f"  {'─'*4}  {'─'*14}  {'─'*10}  {'─'*10}  {'─'*6}")

    entry   = {}
    base_lv = None

    for k in MEASURE_AT:
        y = decode_fractal_to_iter(frac_path, k)
        lv = laplacian_variance(y)
        if base_lv is None and lv > 0:
            base_lv = lv

        if base_lv is None or base_lv == 0:
            rel, dev, tag = 0.0, 0.0, "(blank canvas)"
        elif lv == base_lv:
            rel, dev, tag = 1.0, 0.0, "(baseline)"
        else:
            rel       = lv / base_lv
            predicted = (0.25) ** k
            dev       = abs(rel - predicted) / predicted * 100 if k > 0 else 0.0
            tag       = f"{dev:5.1f}%"

        print(f"  {k:>4}  {lv:>14.2f}  {rel:>10.6f}  {predicted:>10.6f}  {tag:>6}")

        entry[k] = {
            "laplacian_variance":   round(lv, 2),
            "relative_sharpness":   round(rel, 6),
            "predicted_quarter_k":  round(predicted, 6),
            "deviation_pct":        round(dev, 1),
        }

    all_results[label] = entry
    sharpness_log[label] = {
        "timestamp":    datetime.now().isoformat(),
        "frac_path":    frac_path,
        "measurements": entry,
    }

# ── save log ──────────────────────────────────────────────────
with open(LOG_PATH, "w") as f:
    json.dump(sharpness_log, f, indent=2)
print(f"\n  Log saved → {LOG_PATH}")

# ── fit-quality summary ───────────────────────────────────────
print("\n" + "=" * 65)
print("  FIT QUALITY SUMMARY  (deviation from (1/4)^k geometric series)")
print("=" * 65)
print(f"  {'Image':<38}  {'k=1':>6}  {'k=3':>6}  {'k=5':>6}")
print(f"  {'─'*38}  {'─'*6}  {'─'*6}  {'─'*6}")
for label, data in all_results.items():
    d = {k: v["deviation_pct"] for k, v in data.items()}
    print(f"  {label:<38}  {str(d.get(1,'—'))+'%':>6}  "
          f"{str(d.get(3,'—'))+'%':>6}  {str(d.get(5,'—'))+'%':>6}")

print("\n  Low deviation → (1/4)^k approximation holds (smooth images)")
print("  High deviation → model breaks (high-entropy images)")



  SHARPNESS DECAY MEASUREMENT
  2026-03-12 09:54:13

  ── Street scene (smooth) ──
     k    Lap.Variance   Rel.Sharp     (1/4)^k    Dev%
  ────  ──────────────  ──────────  ──────────  ──────
Loaded v7  1368x768 (1366x768)  colour=LOSSY  residual=no


NameError: name 'predicted' is not defined

In [ ]:

# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL B — JPEG COMPARISON  (all 5 benchmark images)         ║
# ║  Requires: original image files + .frac files already saved ║
# ║  Set the BENCHMARK_IMAGES paths below.                      ║
# ╚══════════════════════════════════════════════════════════════╝

import cv2, os, io
import numpy as np
from PIL import Image as PILImage
from skimage.metrics import structural_similarity as ssim_fn

# ── SET THESE ─────────────────────────────────────────────────
BENCHMARK_IMAGES = [
    {"label": "Street scene",
     "orig":  "/content/street scene png var.png",    # path to original image
     "frac_kb": 75.7, "frac_psnr": 38.23, "frac_ssim": 0.983},

    {"label": "Architectural render",
     "orig":  "/content/simple cube png variant.png",
     "frac_kb": 12.8, "frac_psnr": 40.57, "frac_ssim": 0.990},

    {"label": "CCTV footage",
     "orig":  "/content/cctv png variant.png",
     "frac_kb": 112.6, "frac_psnr": 26.01, "frac_ssim": 0.880},

    {"label": "Brick wall",
     "orig":  "/content/brick wall png var.png",
     "frac_kb": 239.9, "frac_psnr": 25.48, "frac_ssim": 0.672},

    {"label": "Forest canopy",
     "orig":  "/content/canopy image shrunk png.png",
     "frac_kb": 172.1, "frac_psnr": 20.70, "frac_ssim": 0.727},
]
# ─────────────────────────────────────────────────────────────

def img_metrics(orig_rgb, recon_rgb):
    o = orig_rgb.astype(np.float64)
    r = recon_rgb.astype(np.float64)
    mse = np.mean((o - r) ** 2)
    if mse == 0:
        return float('inf'), 1.0
    psnr = 10 * np.log10(255.0 ** 2 / mse)
    og = cv2.cvtColor(orig_rgb,  cv2.COLOR_RGB2GRAY)
    rg = cv2.cvtColor(recon_rgb, cv2.COLOR_RGB2GRAY)
    s  = ssim_fn(og, rg, data_range=255)
    return round(psnr, 2), round(float(s), 3)

def jpeg_encode_decode(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.jpg', img_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec_bgr = cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR)
    dec_rgb = cv2.cvtColor(dec_bgr, cv2.COLOR_BGR2RGB)
    return dec_rgb, len(buf) / 1024.0

def find_nearest_jpeg_quality(img_rgb, target_kb, step=1):
    """Return quality whose file size is closest to target_kb."""
    best_q, best_diff = 1, float('inf')
    for q in range(1, 96, step):
        _, kb = jpeg_encode_decode(img_rgb, q)
        diff = abs(kb - target_kb)
        if diff < best_diff:
            best_diff, best_q = diff, q
    return best_q

def png_size_kb(img_rgb):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.png', img_bgr)
    return round(len(buf) / 1024, 1)

def lanczos_2x_psnr(orig_rgb):
    """
    Downscale to half resolution with area filter, save as JPEG q=85,
    upscale back to original with Lanczos — mimics 'store half-res JPEG
    and upscale' as the baseline for resolution-independence comparison.
    Returns (file_size_kb, psnr, ssim).
    """
    H, W   = orig_rgb.shape[:2]
    small  = cv2.resize(orig_rgb, (W // 2, H // 2),
                        interpolation=cv2.INTER_AREA)
    small_dec, kb = jpeg_encode_decode(small, 85)
    upscaled = cv2.resize(small_dec, (W, H),
                          interpolation=cv2.INTER_LANCZOS4)
    p, s = img_metrics(orig_rgb, upscaled)
    return round(kb, 1), p, s

# ── main loop ─────────────────────────────────────────────────
print("=" * 78)
print("  JPEG COMPARISON — ALL BENCHMARK IMAGES")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 78)

jpeg_results = {}

for cfg in BENCHMARK_IMAGES:
    label     = cfg["label"]
    orig_path = cfg["orig"]
    frac_kb   = cfg["frac_kb"]

    if not os.path.exists(orig_path):
        print(f"\n  [SKIP] {label} — not found: {orig_path}")
        continue

    orig_rgb  = np.array(PILImage.open(orig_path).convert('RGB'))
    H, W      = orig_rgb.shape[:2]

    print(f"\n  ── {label}  ({W}×{H}) ──")

    # PNG baseline
    png_kb = png_size_kb(orig_rgb)
    print(f"  PNG (lossless)     : {png_kb:>7.1f} KB  (lossless reference)")

    # Fractal (from experiment log)
    print(f"  Fractal v7 (ours)  : {frac_kb:>7.1f} KB  "
          f"PSNR={cfg['frac_psnr']} dB  SSIM={cfg['frac_ssim']}")

    # JPEG at nearest matched file size
    best_q           = find_nearest_jpeg_quality(orig_rgb, frac_kb)
    jpeg_dec, jpeg_kb = jpeg_encode_decode(orig_rgb, best_q)
    j_psnr, j_ssim   = img_metrics(orig_rgb, jpeg_dec)
    print(f"  JPEG q={best_q:<2} (matched)  : {jpeg_kb:>7.1f} KB  "
          f"PSNR={j_psnr} dB  SSIM={j_ssim}")

    # Lanczos 2× baseline (resolution-independence comparison)
    lz_kb, lz_psnr, lz_ssim = lanczos_2x_psnr(orig_rgb)
    print(f"  JPEG q=85 + Lanczos 2×: {lz_kb:>5.1f} KB  "
          f"PSNR={lz_psnr} dB  SSIM={lz_ssim}  (half-res JPEG upscaled)")

    delta = round(cfg['frac_psnr'] - lz_psnr, 2)
    winner = "fractal wins" if delta > 0 else "lanczos wins"
    print(f"  Resolution-independence ΔPSNR: {delta:+.2f} dB  ({winner})")

    jpeg_results[label] = {
        "resolution":         f"{W}×{H}",
        "png_kb":             png_kb,
        "frac_kb":            frac_kb,
        "frac_psnr":          cfg["frac_psnr"],
        "frac_ssim":          cfg["frac_ssim"],
        "jpeg_quality":       best_q,
        "jpeg_kb":            round(jpeg_kb, 1),
        "jpeg_psnr":          j_psnr,
        "jpeg_ssim":          j_ssim,
        "lanczos_2x_kb":      lz_kb,
        "lanczos_2x_psnr":    lz_psnr,
        "lanczos_2x_ssim":    lz_ssim,
        "res_indep_delta_db": delta,
    }

# ── save results ──────────────────────────────────────────────
with open("jpeg_comparison_log.json", "w") as f:
    json.dump(jpeg_results, f, indent=2)
print("\n  Log saved → jpeg_comparison_log.json")

# ── paper-ready summary table ─────────────────────────────────
print("\n" + "=" * 78)
print("  PAPER TABLE — JPEG comparison across all images")
print("=" * 78)
print(f"  {'Image':<24} {'Frac':>7} {'Frac PSNR':>10} {'JPEG':>7} "
      f"{'JPEG PSNR':>10} {'Lz2× PSNR':>10} {'ΔPSNR':>7} {'PNG':>8}")
print(f"  {'─'*24} {'─'*7} {'─'*10} {'─'*7} {'─'*10} {'─'*10} {'─'*7} {'─'*8}")
for label, r in jpeg_results.items():
    print(f"  {label:<24} {r['frac_kb']:>6.1f}K {r['frac_psnr']:>10.2f} "
          f"{r['jpeg_kb']:>6.1f}K {r['jpeg_psnr']:>10.2f} "
          f"{r['lanczos_2x_psnr']:>10.2f} {r['res_indep_delta_db']:>+7.2f} "
          f"{r['png_kb']:>7.1f}K")

print("\n  ΔPSNR = Fractal PSNR − JPEG+Lanczos 2× PSNR")
print("  Positive Δ = fractal resolution-independence beats Lanczos upscaling.")
print("  JPEG matched to nearest file size. Lanczos baseline uses JPEG q=85 at half-res.")

## Cell 16 — 2x / 4x Upscale

In [ ]:
for scale in [2, 4]:
    print(f'\n── {scale}x Upscale ──')
    up = decode_and_upscale(y_tf, cb_enc, cr_enc,
                             (H_pad,W_pad), (orig_H,orig_W),
                             scale=scale, n_i=12)
    print(f'  {orig_W}x{orig_H} -> {up.shape[1]}x{up.shape[0]}')
    print(f'  File: {os.path.getsize(OUTPUT_PATH)/1024:.1f} KB (unchanged)')
    display(Image.fromarray(up))


## Cell 17 — Rate-Distortion Benchmark vs JPEG

In [ ]:
def run_rate_distortion(image_path, thresholds=None, n_iter=10):
    import io
    from scipy.ndimage import uniform_filter
    if thresholds is None: thresholds = [5, 15, 30, 50, 80, 120, 200]

    img_raw = np.array(Image.open(image_path).convert('RGB'))
    img, oH, oW = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]
    ycbcr = rgb_to_ycbcr(img)
    orig  = img_raw

    base_cfg = analyse_image(ycbcr[:,:,0], TIME_BUDGET_SECONDS, False)
    base_cfg['domain_step']  = DOMAIN_STEP
    base_cfg['colour_mode']  = 'LOSSY'
    ds, _ = build_domain_stack(
        ycbcr[:,:,0].astype(np.float32), DOMAIN_STEP)

    def ssim_c(x, y, L=255, k1=0.01, k2=0.03):
        w=11; mx=uniform_filter(x,w); my=uniform_filter(y,w)
        mxx=uniform_filter(x*x,w); myy=uniform_filter(y*y,w)
        mxy=uniform_filter(x*y,w)
        sx=mxx-mx**2; sy=myy-my**2; sxy=mxy-mx*my
        C1,C2=(k1*L)**2,(k2*L)**2
        return np.mean(((2*mx*my+C1)*(2*sxy+C2))/
                       ((mx**2+my**2+C1)*(sx+sy+C2)))

    fpts = []
    for thresh in thresholds:
        rc = dict(base_cfg); rc['error_threshold'] = thresh
        ytf, _ = encode_channel_gpu(ycbcr[:,:,0], rc)
        cbe = encode_colour_channel(ycbcr[:,:,1], 'LOSSY', DCT_QUALITY_FACTOR)
        cre = encode_colour_channel(ycbcr[:,:,2], 'LOSSY', DCT_QUALITY_FACTOR)
        tmp = f'/tmp/rd_{thresh}.frac'
        save_compressed(tmp, img.shape[:2], (oH,oW), ytf, cbe, cre, rc)
        sz  = os.path.getsize(tmp) / 1024
        yr  = decode_fractal(ytf, (H_pad,W_pad), n_iter)
        res = crop_to_original(
            ycbcr_to_rgb(np.stack([yr,
                decode_colour_channel(cbe),
                decode_colour_channel(cre)], axis=2)), oH, oW)
        a=orig.astype(np.float64); b=res.astype(np.float64)
        mse=np.mean((a-b)**2); psnr=10*np.log10(255**2/mse)
        ssim=np.mean([ssim_c(a[:,:,c],b[:,:,c]) for c in range(3)])
        fpts.append((sz, psnr, ssim, thresh))
        print(f'  T={thresh:3d}  {sz:.0f}KB  PSNR={psnr:.2f}  SSIM={ssim:.4f}')

    jpts = []
    for q in [5, 10, 20, 30, 50, 70, 85, 95]:
        buf = io.BytesIO()
        Image.fromarray(orig).save(buf, format='JPEG', quality=q)
        sz = buf.tell()/1024; buf.seek(0)
        j  = np.array(Image.open(buf))
        a=orig.astype(np.float64); b=j.astype(np.float64)
        mse=np.mean((a-b)**2); psnr=10*np.log10(255**2/mse)
        ssim=np.mean([ssim_c(a[:,:,c],b[:,:,c]) for c in range(3)])
        jpts.append((sz, psnr, ssim, q))

    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
    ax1.plot([p[0] for p in jpts],[p[1] for p in jpts],'b-o',label='JPEG',lw=2)
    ax1.plot([p[0] for p in fpts],[p[1] for p in fpts],'r-s',label='Fractal',lw=2)
    ax1.set_xlabel('File size (KB)'); ax1.set_ylabel('PSNR (dB)')
    ax1.set_title('Rate-Distortion'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot([p[0] for p in jpts],[p[2] for p in jpts],'b-o',label='JPEG',lw=2)
    ax2.plot([p[0] for p in fpts],[p[2] for p in fpts],'r-s',label='Fractal',lw=2)
    ax2.set_xlabel('File size (KB)'); ax2.set_ylabel('SSIM')
    ax2.set_title('Rate-Distortion SSIM'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/rate_distortion_v7.png', dpi=150, bbox_inches='tight')
    plt.show()
    return fpts, jpts

# Uncomment to run:
# fpts, jpts = run_rate_distortion(IMAGE_PATH)
print('Rate-distortion benchmark defined')
